# Notebook 03 — Race Identity and Source-Key Reconstruction

## Purpose

This notebook investigates how individual races and runner records can be identified reliably within the source data.

The source-provided `race_id` is reused across genuinely different races, so it cannot be treated as a globally unique race identifier. Earlier profiling found that the descriptive grouping:

`date + course + off + race_name`

produces 189,043 provisional apparent races, while adding `horse` produces unique provisional runner groups across the current source. These findings are useful starting points, but neither grouping has yet been accepted as a durable identity rule.

This notebook will test:

- how `race_id` behaves within and across dates;
- whether `date + course + off + race_name` is sufficiently stable as a provisional race key;
- where that descriptive grouping collides or splits apparently identical races;
- whether `horse` is sufficient to identify a runner within a provisional race;
- how duplicate runner numbers, coupled entries and amended records affect runner identity;
- whether repeated or revised source rows exist;
- which source-lineage fields must be retained;
- which surrogate identifiers a later staging layer will require.

The notebook will produce evidence-based candidate race and runner identity rules, document unresolved risks, and recommend source-lineage requirements. It will not design the final target schema.

In [1]:
# Read-only source setup for Notebook 03.
#
# The raw SQLite database is immutable. Opening it with SQLite's URI
# read-only mode prevents this notebook from modifying the source file.

from pathlib import Path
import sqlite3

# Resolve the repository root whether the notebook is run from the project
# root or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_DB = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"

if not SOURCE_DB.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB}")

# mode=ro enforces read-only access at the SQLite connection level.
connection = sqlite3.connect(f"file:{SOURCE_DB}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {SOURCE_DB}")
print("SQLite access: read-only")
print(f"Source table: {SOURCE_TABLE}")
print(f"Data-row predicate: {DATA_ROW_PREDICATE}")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
SQLite access: read-only
Source table: data
Data-row predicate: rowid <> 1


In [2]:
# Establish the headline behaviour of the supplied race_id.
#
# We compare:
# - the number of distinct race_id values;
# - the number of distinct date + race_id combinations;
# - how many race_id values appear on more than one date;
# - the greatest number of dates attached to any single race_id.
#
# This is an initial population-level measurement only. It does not yet tell
# us whether repeated race_id values refer to the same or different races.

import pandas as pd

race_id_summary_sql = f"""
WITH race_id_by_date AS (
    SELECT
        CAST(race_id AS TEXT) AS race_id,
        date,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        CAST(race_id AS TEXT),
        date
),
race_id_date_counts AS (
    SELECT
        race_id,
        COUNT(*) AS distinct_dates
    FROM race_id_by_date
    GROUP BY race_id
)
SELECT
    (SELECT COUNT(DISTINCT CAST(race_id AS TEXT))
     FROM {SOURCE_TABLE}
     WHERE {DATA_ROW_PREDICATE}) AS distinct_race_ids,

    (SELECT COUNT(*)
     FROM race_id_by_date) AS distinct_race_id_date_pairs,

    (SELECT COUNT(*)
     FROM race_id_date_counts
     WHERE distinct_dates > 1) AS race_ids_used_on_multiple_dates,

    (SELECT MAX(distinct_dates)
     FROM race_id_date_counts) AS maximum_dates_for_one_race_id
"""

race_id_summary = pd.read_sql_query(race_id_summary_sql, connection)
race_id_summary

,distinct_race_ids,distinct_race_id_date_pairs,race_ids_used_on_multiple_dates,maximum_dates_for_one_race_id
0,188782,189035,206,5


In [3]:
# Profile the 206 race_id values that occur on more than one date.
#
# This shows the distribution of reuse: how many identifiers appear on
# exactly two, three, four or five distinct dates. Runner-row counts are
# included only as descriptive evidence at this stage.

race_id_reuse_distribution_sql = f"""
WITH race_id_profiles AS (
    SELECT
        CAST(race_id AS TEXT) AS race_id,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY CAST(race_id AS TEXT)
)
SELECT
    distinct_dates,
    COUNT(*) AS race_id_count,
    SUM(runner_rows) AS runner_rows
FROM race_id_profiles
WHERE distinct_dates > 1
GROUP BY distinct_dates
ORDER BY distinct_dates
"""

race_id_reuse_distribution = pd.read_sql_query(
    race_id_reuse_distribution_sql,
    connection,
)

race_id_reuse_distribution

,distinct_dates,race_id_count,runner_rows
0,2,164,3080
1,3,38,1109
2,4,3,137
3,5,1,45


In [4]:
# Inspect every date-specific use of the most heavily reused race_id values.
#
# We show the identifiers used on four or five dates first because they provide
# the clearest evidence about the nature of race_id reuse. Each row represents
# one date-specific descriptive race group, with its runner-row count.

heavily_reused_race_ids_sql = f"""
WITH race_id_date_counts AS (
    SELECT
        CAST(race_id AS TEXT) AS race_id,
        COUNT(DISTINCT date) AS distinct_dates
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY CAST(race_id AS TEXT)
),
selected_ids AS (
    SELECT race_id, distinct_dates
    FROM race_id_date_counts
    WHERE distinct_dates >= 4
)
SELECT
    s.distinct_dates,
    CAST(d.race_id AS TEXT) AS race_id,
    d.date,
    d.course,
    d.off,
    d.race_name,
    COUNT(*) AS runner_rows
FROM {SOURCE_TABLE} AS d
JOIN selected_ids AS s
    ON CAST(d.race_id AS TEXT) = s.race_id
WHERE {DATA_ROW_PREDICATE}
GROUP BY
    s.distinct_dates,
    CAST(d.race_id AS TEXT),
    d.date,
    d.course,
    d.off,
    d.race_name
ORDER BY
    s.distinct_dates DESC,
    race_id,
    d.date,
    d.course,
    d.off,
    d.race_name
"""

heavily_reused_race_ids = pd.read_sql_query(
    heavily_reused_race_ids_sql,
    connection,
)

heavily_reused_race_ids

,distinct_dates,race_id,date,course,off,race_name,runner_rows
0,5,650000,2016-05-07,Wexford (IRE),5:25,Owenavarragh (Q.R.) Maiden,10
1,5,650000,2016-05-25,Kempton (AW),7:20,£10 Free Bet At 32RedSport.com Handicap,13
2,5,650000,2016-05-27,Bath,4:00,Bristol Airport Handicap,7
3,5,650000,2016-06-01,Chelmsford (AW),9:10,CCR Ladies Day 16th June Handicap,8
4,5,650000,2016-08-05,Tipperary (IRE),4:55,Anglo Printers 1890 624 624 Nursery Handicap (...,7
5,4,629000,2015-02-28,Caulfield (AUS),4:55,italktravel Futurity Stakes (3yo+) (Turf),8
6,4,629000,2015-06-14,Chantilly (FR),4:00,Prix du Lys Longines (3yo) (Turf),7
7,4,629000,2015-07-02,Perth,2:30,Strongbow Dark Fruit Maiden Hurdle (Strongbow ...,10
8,4,629000,2015-07-04,Nottingham,6:15,Goldeva Maiden Auction Fillies Stakes (Plus 10...,12
9,4,630000,2015-06-28,Saint-Cloud (FR),5:20,Prix de Pau (Handicap) (3yo) (Turf),12


In [5]:
# Test whether race_id + date identifies exactly one descriptive race.
#
# For each race_id/date pair, count distinct combinations of:
#     course + off + race_name
#
# Any pair with more than one descriptive group shows that adding date to the
# supplied race_id is still insufficient to identify one race reliably.

race_id_date_collisions_sql = f"""
WITH descriptive_groups AS (
    SELECT
        CAST(race_id AS TEXT) AS race_id,
        date,
        course,
        off,
        race_name,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        CAST(race_id AS TEXT),
        date,
        course,
        off,
        race_name
),
collision_pairs AS (
    SELECT
        race_id,
        date,
        COUNT(*) AS descriptive_group_count,
        SUM(runner_rows) AS total_runner_rows
    FROM descriptive_groups
    GROUP BY
        race_id,
        date
    HAVING COUNT(*) > 1
)
SELECT
    c.race_id,
    c.date,
    c.descriptive_group_count,
    c.total_runner_rows,
    d.course,
    d.off,
    d.race_name,
    d.runner_rows
FROM collision_pairs AS c
JOIN descriptive_groups AS d
    ON d.race_id = c.race_id
   AND d.date = c.date
ORDER BY
    c.descriptive_group_count DESC,
    c.race_id,
    c.date,
    d.course,
    d.off,
    d.race_name
"""

race_id_date_collisions = pd.read_sql_query(
    race_id_date_collisions_sql,
    connection,
)

race_id_date_collisions

,race_id,date,descriptive_group_count,total_runner_rows,course,off,race_name,runner_rows
0,616900,2015-01-05,2,16,Musselburgh,1:30,Alex Donaldson Handsome Handicap Chase,7
1,616900,2015-01-05,2,16,Thurles (IRE),1:55,Thurles Racecourse Handicap Chase,9
2,620000,2015-03-01,2,23,Gavea (BRZ),7:40,Grande Premio Diana (3yo Fillies) (Turf),14
3,620000,2015-03-01,2,23,Navan (IRE),3:20,Garlow Cross Handicap Hurdle,9
4,620001,2015-03-01,2,19,Gavea (BRZ),6:30,Grande Premio Francisco Eduardo de Paula Macha...,11
5,620001,2015-03-01,2,19,Navan (IRE),3:50,Follow Navan On Facebook Beginners Chase,8
6,622000,2015-03-28,2,22,Meydan (UAE),2:30,Al Quoz Sprint Empowered By IPIC (Turf),16
7,622000,2015-03-28,2,22,Santa Anita (USA),11:05,Tokyo City Cup Stakes (4yo+) (Dirt),6
8,630000,2015-07-18,2,23,Cartmel,5:00,totepool Cumbria Crystal Trophy Handicap Hurdle,16
9,630000,2015-07-18,2,23,Newmarket (July),2:15,Newsells Park Stud Stakes (Registered as The A...,7


In [6]:
# Test whether one provisional descriptive race group contains multiple race_id values.
#
# The provisional race grouping established in Notebook 02 is:
#     date + course + off + race_name
#
# If a descriptive group contains more than one race_id, possible explanations
# include duplicated or amended source records, reused descriptions, or a
# collision in the provisional grouping. This query only identifies candidates;
# it does not decide which explanation applies.

descriptive_group_race_id_collisions_sql = f"""
WITH descriptive_group_profiles AS (
    SELECT
        date,
        course,
        off,
        race_name,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT CAST(race_id AS TEXT)) AS distinct_race_ids
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off,
        race_name
    HAVING COUNT(DISTINCT CAST(race_id AS TEXT)) > 1
)
SELECT
    p.date,
    p.course,
    p.off,
    p.race_name,
    p.runner_rows,
    p.distinct_race_ids,
    CAST(d.race_id AS TEXT) AS race_id,
    COUNT(*) AS rows_for_race_id
FROM descriptive_group_profiles AS p
JOIN {SOURCE_TABLE} AS d
    ON d.date = p.date
   AND d.course = p.course
   AND d.off = p.off
   AND d.race_name = p.race_name
WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
GROUP BY
    p.date,
    p.course,
    p.off,
    p.race_name,
    p.runner_rows,
    p.distinct_race_ids,
    CAST(d.race_id AS TEXT)
ORDER BY
    p.date,
    p.course,
    p.off,
    p.race_name,
    race_id
"""

descriptive_group_race_id_collisions = pd.read_sql_query(
    descriptive_group_race_id_collisions_sql,
    connection,
)

print(
    "Provisional descriptive groups containing multiple race_id values:",
    descriptive_group_race_id_collisions[
        ["date", "course", "off", "race_name"]
    ].drop_duplicates().shape[0],
)

descriptive_group_race_id_collisions

Provisional descriptive groups containing multiple race_id values: 0


,date,course,off,race_name,runner_rows,distinct_race_ids,race_id,rows_for_race_id


In [7]:
# Find date + course + off combinations associated with multiple race names.
#
# A race would normally be expected to have one advertised off-time at one
# course on one date. Multiple race_name values within that combination could
# indicate:
# - an amended or differently formatted race name;
# - duplicated source versions;
# - an off-time collision between genuinely separate races;
# - or another source anomaly.
#
# This query identifies candidates only. It does not yet classify them.

same_slot_multiple_names_sql = f"""
WITH slot_profiles AS (
    SELECT
        date,
        course,
        off,
        COUNT(DISTINCT race_name) AS distinct_race_names,
        COUNT(DISTINCT CAST(race_id AS TEXT)) AS distinct_race_ids,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off
    HAVING COUNT(DISTINCT race_name) > 1
)
SELECT
    p.date,
    p.course,
    p.off,
    p.distinct_race_names,
    p.distinct_race_ids,
    p.runner_rows,
    d.race_name,
    CAST(d.race_id AS TEXT) AS race_id,
    COUNT(*) AS rows_for_name
FROM slot_profiles AS p
JOIN {SOURCE_TABLE} AS d
    ON d.date = p.date
   AND d.course = p.course
   AND d.off = p.off
WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
GROUP BY
    p.date,
    p.course,
    p.off,
    p.distinct_race_names,
    p.distinct_race_ids,
    p.runner_rows,
    d.race_name,
    CAST(d.race_id AS TEXT)
ORDER BY
    p.date,
    p.course,
    p.off,
    race_id,
    d.race_name
"""

same_slot_multiple_names = pd.read_sql_query(
    same_slot_multiple_names_sql,
    connection,
)

print(
    "Date/course/off combinations containing multiple race names:",
    same_slot_multiple_names[
        ["date", "course", "off"]
    ].drop_duplicates().shape[0],
)

same_slot_multiple_names

Date/course/off combinations containing multiple race names: 0


,date,course,off,distinct_race_names,distinct_race_ids,runner_rows,race_name,race_id,rows_for_name


In [8]:
# Find horses assigned to more than one provisional race at the same course
# on the same date.
#
# A horse would not ordinarily run in two separate races at one meeting.
# Therefore, the same horse appearing under multiple off-times or race names
# on the same date and course is a useful candidate signal for:
# - duplicated or amended race records;
# - descriptive splitting of one real race;
# - or an exceptional source anomaly.
#
# This remains a candidate-finding test only. Horse-name reuse or malformed
# records could produce false positives and must be inspected separately.

same_meeting_horse_repetitions_sql = f"""
WITH horse_race_membership AS (
    SELECT
        date,
        course,
        horse,
        off,
        race_name,
        CAST(race_id AS TEXT) AS race_id,
        COUNT(*) AS rows_in_group
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(horse AS TEXT)) <> ''
    GROUP BY
        date,
        course,
        horse,
        off,
        race_name,
        CAST(race_id AS TEXT)
),
repeated_horses AS (
    SELECT
        date,
        course,
        horse,
        COUNT(*) AS provisional_race_count
    FROM horse_race_membership
    GROUP BY
        date,
        course,
        horse
    HAVING COUNT(*) > 1
)
SELECT
    r.date,
    r.course,
    r.horse,
    r.provisional_race_count,
    m.off,
    m.race_name,
    m.race_id,
    m.rows_in_group
FROM repeated_horses AS r
JOIN horse_race_membership AS m
    ON m.date = r.date
   AND m.course = r.course
   AND m.horse = r.horse
ORDER BY
    r.provisional_race_count DESC,
    r.date,
    r.course,
    r.horse,
    m.off,
    m.race_name
"""

same_meeting_horse_repetitions = pd.read_sql_query(
    same_meeting_horse_repetitions_sql,
    connection,
)

print(
    "Horses appearing in multiple provisional races at the same meeting:",
    same_meeting_horse_repetitions[
        ["date", "course", "horse"]
    ].drop_duplicates().shape[0],
)

same_meeting_horse_repetitions

Horses appearing in multiple provisional races at the same meeting: 0


,date,course,horse,provisional_race_count,off,race_name,race_id,rows_in_group


In [9]:
# Test whether horse identifies one runner record within a provisional race.
#
# The provisional race grouping is:
#     date + course + off + race_name
#
# Any group containing the same horse more than once would mean that horse
# alone is insufficient as a provisional runner identifier. Possible causes
# could include duplicate source rows, amended records, coupled entries with
# ambiguous naming, or another source anomaly.

duplicate_horse_within_race_sql = f"""
WITH runner_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        horse,
        COUNT(*) AS source_rows,
        COUNT(DISTINCT CAST(race_id AS TEXT)) AS distinct_race_ids,
        COUNT(DISTINCT CAST(num AS TEXT)) AS distinct_runner_numbers
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(horse AS TEXT)) <> ''
    GROUP BY
        date,
        course,
        off,
        race_name,
        horse
    HAVING COUNT(*) > 1
)
SELECT
    date,
    course,
    off,
    race_name,
    horse,
    source_rows,
    distinct_race_ids,
    distinct_runner_numbers
FROM runner_groups
ORDER BY
    source_rows DESC,
    date,
    course,
    off,
    race_name,
    horse
"""

duplicate_horse_within_race = pd.read_sql_query(
    duplicate_horse_within_race_sql,
    connection,
)

print(
    "Provisional race + horse groups containing multiple source rows:",
    len(duplicate_horse_within_race),
)

duplicate_horse_within_race

Provisional race + horse groups containing multiple source rows: 0


,date,course,off,race_name,horse,source_rows,distinct_race_ids,distinct_runner_numbers


In [10]:
# Test whether runner number is unique within each provisional race.
#
# The source field `num` may represent a racecard number rather than a unique
# participant identifier. Coupled entries in some jurisdictions can share a
# betting number, and malformed or duplicated values may also occur.
#
# Blank runner numbers are excluded from this test because they represent
# missing values rather than an attempted identifier.

duplicate_num_within_race_sql = f"""
WITH numbered_runner_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        CAST(num AS TEXT) AS num,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT horse) AS distinct_horses,
        COUNT(DISTINCT CAST(race_id AS TEXT)) AS distinct_race_ids
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(num AS TEXT)) <> ''
    GROUP BY
        date,
        course,
        off,
        race_name,
        CAST(num AS TEXT)
    HAVING COUNT(*) > 1
)
SELECT
    date,
    course,
    off,
    race_name,
    num,
    runner_rows,
    distinct_horses,
    distinct_race_ids
FROM numbered_runner_groups
ORDER BY
    runner_rows DESC,
    date,
    course,
    off,
    race_name,
    num
"""

duplicate_num_within_race = pd.read_sql_query(
    duplicate_num_within_race_sql,
    connection,
)

print(
    "Provisional race + runner-number groups containing multiple rows:",
    len(duplicate_num_within_race),
)

duplicate_num_within_race

Provisional race + runner-number groups containing multiple rows: 700


,date,course,off,race_name,num,runner_rows,distinct_horses,distinct_race_ids
0,2026-03-21,Chukyo,06:20,Chunichi Sports Sho Falcon Stakes (Turf),0,17,17,1
1,2016-12-07,Lyon-La Soie (FR),5:10,Prix Du Haras Des Chataigniers (Claimer) (All-...,0,16,16,1
2,2018-11-04,Kyoto (JPN),6:01,JBC Sprint (Local (Dirt),0,16,16,1
3,2026-03-21,Nakayama,06:45,Flower Cup (Fillies) (Turf),0,16,16,1
4,2018-08-16,Mombetsu (JPN),11:07,Breeders Gold Cup (Local (Dirt),0,15,15,1
...,...,...,...,...,...,...,...,...
695,2024-05-25,San Isidro (ARG),9:45,Gran Premio 25 De Mayo - Copa Dr Enrique Olive...,6,2,2,1
696,2024-06-29,San Isidro (ARG),8:20,Gran Premio Estrellas Distaff - Copa Sr Alejan...,4,2,2,1
697,2024-08-04,Del Mar (USA),2:35,Clement L Hirsch Stakes presented by Oak Tree ...,8,2,2,1
698,2024-11-19,La Plata (ARG),10:10,Gran Premio Dardo Rocha (3yo+) (Dirt),12,2,2,1


In [11]:
# Separate duplicate runner-number groups into zero and non-zero values.
#
# This distinguishes races where `num = 0` behaves like a missing-value
# sentinel from races where a non-zero betting or racecard number is shared
# by multiple horses, potentially representing coupled entries.

duplicate_num_profile_sql = f"""
WITH duplicate_number_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT)) AS num_text,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT horse) AS distinct_horses
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(num AS TEXT)) <> ''
    GROUP BY
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT))
    HAVING COUNT(*) > 1
)
SELECT
    CASE
        WHEN num_text IN ('0', '0.0') THEN 'zero'
        ELSE 'non-zero'
    END AS number_category,
    COUNT(*) AS duplicate_groups,
    SUM(runner_rows) AS runner_rows,
    MIN(runner_rows) AS minimum_rows_per_group,
    MAX(runner_rows) AS maximum_rows_per_group
FROM duplicate_number_groups
GROUP BY number_category
ORDER BY number_category DESC
"""

duplicate_num_profile = pd.read_sql_query(
    duplicate_num_profile_sql,
    connection,
)

duplicate_num_profile

,number_category,duplicate_groups,runner_rows,minimum_rows_per_group,maximum_rows_per_group
0,zero,177,1170,2,17
1,non-zero,523,1084,2,4


In [12]:
# Inspect the largest non-zero runner-number collisions.
#
# Each returned row is an individual horse. The group-level columns show how
# many horses share the same non-zero `num` within the provisional race.
#
# This should reveal whether collisions represent coupled entries, number
# suffixes stored elsewhere, or another source convention.

nonzero_duplicate_num_examples_sql = f"""
WITH duplicate_number_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT)) AS num,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(num AS TEXT)) NOT IN ('', '0', '0.0')
    GROUP BY
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT))
    HAVING COUNT(*) > 1
),
ranked_groups AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY
                runner_rows DESC,
                date,
                course,
                off,
                race_name,
                num
        ) AS group_rank
    FROM duplicate_number_groups
)
SELECT
    g.group_rank,
    g.runner_rows AS horses_sharing_number,
    d.date,
    d.course,
    d.off,
    d.race_name,
    g.num,
    d.horse,
    d.pos,
    d.draw,
    d.sp
FROM ranked_groups AS g
JOIN {SOURCE_TABLE} AS d
    ON d.date = g.date
   AND d.course = g.course
   AND d.off = g.off
   AND d.race_name = g.race_name
   AND TRIM(CAST(d.num AS TEXT)) = g.num
WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
  AND g.group_rank <= 20
ORDER BY
    g.group_rank,
    d.rowid
"""

nonzero_duplicate_num_examples = pd.read_sql_query(
    nonzero_duplicate_num_examples_sql,
    connection,
)

nonzero_duplicate_num_examples

,group_rank,horses_sharing_number,date,course,off,race_name,num,horse,pos,draw,sp
0,1,4,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,London Show (URU),8,12,162/10
1,1,4,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,Litle Eatly (BRZ),12,11,162/10
2,1,4,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,Mendel Sweet (BRZ),14,13,162/10
3,1,4,2018-12-08,San Isidro (ARG),9:55,Copa de Plata (3yo+ Fillies & Mares) (Turf),11,Linda Le (BRZ),17,10,162/10
4,2,3,2015-01-11,Monterrico (PER),10:25,Premio Enrique Meiggs (3yo+) (Dirt),10,Good Luck Keny (ARG),2,10,
...,...,...,...,...,...,...,...,...,...,...,...
56,19,3,2017-09-24,La Plata (ARG),10:05,Seleccion de Potrancas (3yo Fillies) (Dirt),3,Halo Holiday (ARG),1,3,13/10F
57,19,3,2017-09-24,La Plata (ARG),10:05,Seleccion de Potrancas (3yo Fillies) (Dirt),3,Marquesa Nistel (ARG),9,4,13/10F
58,20,3,2017-10-14,San Isidro (ARG),8:00,Gran Premio Suipacha (3yo+) (Turf),7,Sunny Point (ARG),3,10,73/20
59,20,3,2017-10-14,San Isidro (ARG),8:00,Gran Premio Suipacha (3yo+) (Turf),7,Ultra Urbano (ARG),4,11,73/20


In [13]:
# Profile non-zero duplicate runner numbers by course and starting-price pattern.
#
# For each provisional race + non-zero runner-number group, count:
# - the number of horses sharing the number;
# - the number of distinct non-blank starting prices;
# - whether all rows share one recorded starting price.
#
# The course summary then shows where these likely coupled entries are
# concentrated and how consistently they share betting odds.

nonzero_duplicate_num_course_profile_sql = f"""
WITH duplicate_number_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT)) AS num,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT CASE
                WHEN TRIM(CAST(sp AS TEXT)) <> ''
                THEN TRIM(CAST(sp AS TEXT))
            END
        ) AS distinct_nonblank_sp
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(num AS TEXT)) NOT IN ('', '0', '0.0')
    GROUP BY
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT))
    HAVING COUNT(*) > 1
)
SELECT
    course,
    COUNT(*) AS duplicate_number_groups,
    SUM(runner_rows) AS runner_rows,
    SUM(
        CASE WHEN distinct_nonblank_sp = 1 THEN 1 ELSE 0 END
    ) AS groups_with_one_nonblank_sp,
    SUM(
        CASE WHEN distinct_nonblank_sp = 0 THEN 1 ELSE 0 END
    ) AS groups_with_no_recorded_sp,
    SUM(
        CASE WHEN distinct_nonblank_sp > 1 THEN 1 ELSE 0 END
    ) AS groups_with_multiple_sp_values,
    MAX(runner_rows) AS maximum_horses_sharing_number
FROM duplicate_number_groups
GROUP BY course
ORDER BY
    duplicate_number_groups DESC,
    runner_rows DESC,
    course
"""

nonzero_duplicate_num_course_profile = pd.read_sql_query(
    nonzero_duplicate_num_course_profile_sql,
    connection,
)

print(
    "Courses represented:",
    len(nonzero_duplicate_num_course_profile),
)

nonzero_duplicate_num_course_profile

Courses represented: 29


,course,duplicate_number_groups,runner_rows,groups_with_one_nonblank_sp,groups_with_no_recorded_sp,groups_with_multiple_sp_values,maximum_horses_sharing_number
0,San Isidro (ARG),109,230,97,6,6,4
1,Cidade Jardim (BRZ),90,180,89,0,1,2
2,Gavea (BRZ),77,154,74,0,3,2
3,Monterrico (PER),71,155,12,59,0,3
4,Palermo (ARG),54,109,53,0,1,3
5,Maronas (URU),34,72,34,0,0,3
6,La Plata (ARG),21,44,21,0,0,3
7,Aqueduct (USA),12,25,12,0,0,3
8,Belmont Park (USA),8,16,8,0,0,2
9,Club Hipico de Santiago (CHI),6,15,6,0,0,3


In [14]:
# Inspect every non-zero duplicate runner-number group containing more than
# one recorded starting-price value.
#
# These are exceptions to the dominant coupled-entry pattern. Showing the
# individual rows lets us distinguish genuine numbering conventions from
# possible source errors or unrelated uses of the same runner number.

multiple_sp_duplicate_num_sql = f"""
WITH exceptional_groups AS (
    SELECT
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT)) AS num,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT CASE
                WHEN TRIM(CAST(sp AS TEXT)) <> ''
                THEN TRIM(CAST(sp AS TEXT))
            END
        ) AS distinct_nonblank_sp
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND TRIM(CAST(num AS TEXT)) NOT IN ('', '0', '0.0')
    GROUP BY
        date,
        course,
        off,
        race_name,
        TRIM(CAST(num AS TEXT))
    HAVING COUNT(*) > 1
       AND COUNT(
            DISTINCT CASE
                WHEN TRIM(CAST(sp AS TEXT)) <> ''
                THEN TRIM(CAST(sp AS TEXT))
            END
       ) > 1
)
SELECT
    g.runner_rows AS horses_sharing_number,
    g.distinct_nonblank_sp,
    d.date,
    d.course,
    d.off,
    d.race_name,
    g.num,
    d.horse,
    d.pos,
    d.draw,
    d.sp,
    d.jockey,
    d.trainer
FROM exceptional_groups AS g
JOIN {SOURCE_TABLE} AS d
    ON d.date = g.date
   AND d.course = g.course
   AND d.off = g.off
   AND d.race_name = g.race_name
   AND TRIM(CAST(d.num AS TEXT)) = g.num
WHERE {DATA_ROW_PREDICATE.replace("rowid", "d.rowid")}
ORDER BY
    d.date,
    d.course,
    d.off,
    g.num,
    d.rowid
"""

multiple_sp_duplicate_num = pd.read_sql_query(
    multiple_sp_duplicate_num_sql,
    connection,
)

print(
    "Duplicate-number groups with multiple recorded SP values:",
    multiple_sp_duplicate_num[
        ["date", "course", "off", "race_name", "num"]
    ].drop_duplicates().shape[0],
)

multiple_sp_duplicate_num

Duplicate-number groups with multiple recorded SP values: 20


,horses_sharing_number,distinct_nonblank_sp,date,course,off,race_name,num,horse,pos,draw,sp,jockey,trainer
0,2,2,2015-03-19,Saint-Cloud (FR),2:55,Prix de Juvisy (Claimer) (4yo+) (Turf),4,Dence (FR),10,4,53/1,David Breux,Mlle C Comte
1,2,2,2015-03-19,Saint-Cloud (FR),2:55,Prix de Juvisy (Claimer) (4yo+) (Turf),4,Atomic Bere (FR),6,7,84/10,Frank Panicucci,Mlle C Cardenne
2,2,2,2015-06-21,Gavea (BRZ),8:45,Grande Premio Brasil (3yo+) (Turf),16,Leme Norte (BRZ),11,16,208/10,A Correia,J C Sampaio
3,2,2,2015-06-21,Gavea (BRZ),8:45,Grande Premio Brasil (3yo+) (Turf),16,Quinhao (BRZ),4,17,26/5,A Gulart,M Ferreira
4,2,2,2015-06-27,San Isidro (ARG),7:30,Gran Premio Estrellas Juvenile (2yo Colts) (T...,2,El Atlantico (ARG),4,4,53/20F,Gustavo E Calvente,Alfredo F Gaitan Dassie
5,2,2,2015-06-27,San Isidro (ARG),7:30,Gran Premio Estrellas Juvenile (2yo Colts) (T...,2,Ember Island (ARG),5,1,53/20,Jorge A Ricardo,Alfredo F Gaitan Dassie
6,2,2,2016-03-12,Gavea (BRZ),8:25,Grande Premio Cruzeiro do Sul (3yo) (Turf),11,Million Air (BRZ),8,11,44/1,A Queiroz,M Paulo
7,2,2,2016-03-12,Gavea (BRZ),8:25,Grande Premio Cruzeiro do Sul (3yo) (Turf),11,Umbelievable (BRZ),14,19,79/1,H Fernandes,F Borges
8,2,2,2016-10-08,Cidade Jardim (BRZ),8:30,Grande Premio Jockey Club de Sao Paulo (3yo C...,5,Salto Olimpico (BRZ),1,11,2/1F,V Souza,R Solanes
9,2,2,2016-10-08,Cidade Jardim (BRZ),8:30,Grande Premio Jockey Club de Sao Paulo (3yo C...,5,Olympic Galaxy (BRZ),PU,5,111/10,W Blandi,R Solanes


In [15]:
# Test for exact duplicate source records while excluding SQLite rowid.
#
# We first obtain the physical source-column names from SQLite metadata.
# We then group by every supplied column and count groups occurring more than
# once. Because rowid is not part of the grouping, any result represents rows
# that are exact duplicates in all source fields.
#
# This is deliberately a read-only population-level test. It does not treat
# near-duplicates or amended records as identical.

source_columns = [
    row["name"]
    for row in connection.execute(f"PRAGMA table_info({SOURCE_TABLE})").fetchall()
]

if not source_columns:
    raise RuntimeError(f"No columns found for source table: {SOURCE_TABLE}")

# Quote identifiers defensively in case a source column name conflicts with
# SQL syntax or contains unusual characters.
quoted_columns = [f'"{column.replace(chr(34), chr(34) * 2)}"' for column in source_columns]
grouping_columns_sql = ",\n        ".join(quoted_columns)

exact_duplicate_summary_sql = f"""
WITH exact_row_groups AS (
    SELECT
        {grouping_columns_sql},
        COUNT(*) AS duplicate_count
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        {grouping_columns_sql}
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS exact_duplicate_groups,
    COALESCE(SUM(duplicate_count), 0) AS rows_in_duplicate_groups,
    COALESCE(SUM(duplicate_count - 1), 0) AS excess_duplicate_rows,
    COALESCE(MAX(duplicate_count), 0) AS maximum_copies_of_one_record
FROM exact_row_groups
"""

exact_duplicate_summary = pd.read_sql_query(
    exact_duplicate_summary_sql,
    connection,
)

exact_duplicate_summary

,exact_duplicate_groups,rows_in_duplicate_groups,excess_duplicate_rows,maximum_copies_of_one_record
0,0,0,0,0


In [16]:
# Test whether each provisional race occupies one contiguous block of source rowids.
#
# For a race with N runner rows, a fully contiguous source block will satisfy:
#
#     maximum rowid - minimum rowid + 1 = N
#
# Non-contiguous groups could indicate interleaved races, appended amendments,
# repeated imports or another source-order anomaly.
#
# SQLite rowid is treated only as source-lineage evidence. It is not proposed
# as a durable race or runner business key.

race_rowid_contiguity_sql = f"""
WITH race_rowid_profiles AS (
    SELECT
        date,
        course,
        off,
        race_name,
        COUNT(*) AS runner_rows,
        MIN(rowid) AS first_source_rowid,
        MAX(rowid) AS last_source_rowid,
        MAX(rowid) - MIN(rowid) + 1 AS rowid_span
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        off,
        race_name
)
SELECT
    COUNT(*) AS provisional_races,
    SUM(
        CASE WHEN rowid_span = runner_rows THEN 1 ELSE 0 END
    ) AS contiguous_races,
    SUM(
        CASE WHEN rowid_span <> runner_rows THEN 1 ELSE 0 END
    ) AS noncontiguous_races,
    MAX(rowid_span - runner_rows) AS maximum_internal_rowid_gap
FROM race_rowid_profiles
"""

race_rowid_contiguity = pd.read_sql_query(
    race_rowid_contiguity_sql,
    connection,
)

race_rowid_contiguity

,provisional_races,contiguous_races,noncontiguous_races,maximum_internal_rowid_gap
0,189043,71960,117083,770


In [17]:
# Count the number of separate physical rowid segments occupied by each
# provisional race.
#
# A new segment begins whenever the preceding physical source row belongs to
# a different provisional race. This gives a more direct measure of source
# interleaving than the earlier minimum-to-maximum rowid span.
#
# The result is a population summary only. SQLite rowid remains lineage
# metadata and is not proposed as a business identifier.

race_segment_distribution_sql = f"""
WITH ordered_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        LAG(date) OVER (ORDER BY rowid) AS previous_date,
        LAG(course) OVER (ORDER BY rowid) AS previous_course,
        LAG(off) OVER (ORDER BY rowid) AS previous_off,
        LAG(race_name) OVER (ORDER BY rowid) AS previous_race_name
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
segment_starts AS (
    SELECT
        date,
        course,
        off,
        race_name,
        CASE
            WHEN previous_date = date
             AND previous_course = course
             AND previous_off = off
             AND previous_race_name = race_name
            THEN 0
            ELSE 1
        END AS starts_new_segment
    FROM ordered_rows
),
race_segment_counts AS (
    SELECT
        date,
        course,
        off,
        race_name,
        SUM(starts_new_segment) AS physical_segments
    FROM segment_starts
    GROUP BY
        date,
        course,
        off,
        race_name
)
SELECT
    physical_segments,
    COUNT(*) AS provisional_races
FROM race_segment_counts
GROUP BY physical_segments
ORDER BY physical_segments
"""

race_segment_distribution = pd.read_sql_query(
    race_segment_distribution_sql,
    connection,
)

race_segment_distribution

,physical_segments,provisional_races
0,1,71960
1,2,31063
2,3,35331
3,4,22621
4,5,13309
5,6,6971
6,7,5279
7,8,2023
8,9,410
9,10,66


In [18]:
# Identify the provisional races split across the greatest number of physical
# rowid segments.
#
# We include:
# - total runner rows;
# - number of physical segments;
# - first and last source rowid;
# - total rowid span;
# - supplied race_id.
#
# This will show whether highly fragmented races are isolated anomalies or
# ordinary-sized races whose runners have been systematically interleaved.

highly_segmented_races_sql = f"""
WITH ordered_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        CAST(race_id AS TEXT) AS race_id,
        LAG(date) OVER (ORDER BY rowid) AS previous_date,
        LAG(course) OVER (ORDER BY rowid) AS previous_course,
        LAG(off) OVER (ORDER BY rowid) AS previous_off,
        LAG(race_name) OVER (ORDER BY rowid) AS previous_race_name
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
marked_rows AS (
    SELECT
        *,
        CASE
            WHEN previous_date = date
             AND previous_course = course
             AND previous_off = off
             AND previous_race_name = race_name
            THEN 0
            ELSE 1
        END AS starts_new_segment
    FROM ordered_rows
),
race_profiles AS (
    SELECT
        date,
        course,
        off,
        race_name,
        race_id,
        COUNT(*) AS runner_rows,
        SUM(starts_new_segment) AS physical_segments,
        MIN(source_rowid) AS first_source_rowid,
        MAX(source_rowid) AS last_source_rowid,
        MAX(source_rowid) - MIN(source_rowid) + 1 AS rowid_span
    FROM marked_rows
    GROUP BY
        date,
        course,
        off,
        race_name,
        race_id
)
SELECT
    date,
    course,
    off,
    race_name,
    race_id,
    runner_rows,
    physical_segments,
    first_source_rowid,
    last_source_rowid,
    rowid_span
FROM race_profiles
ORDER BY
    physical_segments DESC,
    rowid_span DESC,
    date,
    course,
    off
LIMIT 30
"""

highly_segmented_races = pd.read_sql_query(
    highly_segmented_races_sql,
    connection,
)

highly_segmented_races

,date,course,off,race_name,race_id,runner_rows,physical_segments,first_source_rowid,last_source_rowid,rowid_span
0,2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,667345,40,11,352418,352969,552
1,2017-02-02,Chantilly (FR),2:40,Prix de lAllee des Merisiers (Handicap) (5yo+)...,668629,16,11,326819,327203,385
2,2023-06-22,Ascot,5:00,Britannia Stakes (Heritage Handicap) (Colts & ...,841945,29,11,1359414,1359775,362
3,2023-05-05,Churchill Downs (USA),10:51,Longines Kentucky Oaks (3yo Fillies) (Main Tr...,840196,14,11,1333782,1334109,328
4,2021-04-10,Aintree,5:15,Randox Grand National Handicap Chase (GBB Race),775757,40,11,991775,991925,151
5,2015-10-12,Windsor,2:30,Ronald McDonald House Charities Nursery Handicap,635299,15,11,125851,125971,121
6,2019-08-16,Curragh (IRE),7:50,Derek OSullivan Memorial Apprentice Handicap,737738,22,11,761766,761882,117
7,2017-07-22,Newbury,3:35,Weatherbys Super Sprint Stakes,669245,23,11,406586,406661,76
8,2022-05-30,Redcar,5:15,Racing Again On Bank Holiday Thursday Amateur ...,811326,14,11,1192996,1193068,73
9,2019-04-05,Aintree,4:05,Randox Health Topham Handicap Chase,724113,27,11,690797,690856,60


In [19]:
# Inspect the physical source ordering of the most fragmented race.
#
# For each runner in the 2017 Grand National, show:
# - its source rowid and runner details;
# - the immediately preceding source row;
# - whether that preceding row belongs to the same provisional race.
#
# This is an observational inspection of source ordering. It does not assume
# that position, runner number or neighbouring rows explain the interleaving.

fragmented_race_order_sql = f"""
WITH source_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        horse,
        num,
        pos
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
target_rows AS (
    SELECT *
    FROM source_rows
    WHERE date = '2017-04-08'
      AND course = 'Aintree'
      AND off = '5:15'
      AND race_name = 'Randox Health Grand National Handicap Chase'
)
SELECT
    t.source_rowid,
    t.num,
    t.pos,
    t.horse,

    p.date AS previous_date,
    p.course AS previous_course,
    p.off AS previous_off,
    p.race_name AS previous_race_name,
    p.horse AS previous_horse,
    p.pos AS previous_pos,

    CASE
        WHEN p.date = t.date
         AND p.course = t.course
         AND p.off = t.off
         AND p.race_name = t.race_name
        THEN 1
        ELSE 0
    END AS previous_row_same_race
FROM target_rows AS t
LEFT JOIN source_rows AS p
    ON p.source_rowid = t.source_rowid - 1
ORDER BY t.source_rowid
"""

fragmented_race_order = pd.read_sql_query(
    fragmented_race_order_sql,
    connection,
)

fragmented_race_order

,source_rowid,num,pos,horse,previous_date,previous_course,previous_off,previous_race_name,previous_horse,previous_pos,previous_row_same_race
0,352418,2,PU,More Of That (IRE),2017-04-08,Aintree,4:20,Ryanair Stayers Liverpool Hurdle,Aux Ptits Soins (FR),8,0
1,352420,4,PU,Perfect Candidate (IRE),2017-04-08,Lingfield (AW),3:10,Betway Insider Handicap,Golly Miss Molly (GB),8,0
2,352421,40,PU,Doctor Harper (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Perfect Candidate (IRE),PU,1
3,352422,26,PU,Bishops Road (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Doctor Harper (IRE),PU,1
4,352423,7,PU,Wounded Warrior (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Bishops Road (IRE),PU,1
5,352424,17,PU,Definitly Red (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Wounded Warrior (IRE),PU,1
6,352425,3,PU,Shantou Flyer (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Definitly Red (IRE),PU,1
7,352426,19,PU,Double Shuffle (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Shantou Flyer (IRE),PU,1
8,352428,32,UR,Raz De Maree (FR),2017-04-08,Aintree,4:20,Ryanair Stayers Liverpool Hurdle,Puffin Billy (IRE),PU,0
9,352429,36,UR,Thunder And Roses (IRE),2017-04-08,Aintree,5:15,Randox Health Grand National Handicap Chase,Raz De Maree (FR),UR,1


In [20]:
# Test whether physical source ordering is associated with finishing position.
#
# We compare adjacent source rows and count how often they share:
# - the same date;
# - the same raw `pos` value;
# - both the same date and the same `pos`;
# - the same provisional race.
#
# We also compare the observed adjacent-pair rates with the overall number of
# physical rows. This is descriptive evidence only; it does not prove the
# original source's exact sorting procedure.

source_order_adjacency_sql = f"""
WITH ordered_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        TRIM(CAST(pos AS TEXT)) AS pos_text,
        course,
        off,
        race_name,

        LAG(date) OVER (ORDER BY rowid) AS previous_date,
        LAG(TRIM(CAST(pos AS TEXT))) OVER (ORDER BY rowid) AS previous_pos_text,
        LAG(course) OVER (ORDER BY rowid) AS previous_course,
        LAG(off) OVER (ORDER BY rowid) AS previous_off,
        LAG(race_name) OVER (ORDER BY rowid) AS previous_race_name
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
adjacent_pairs AS (
    SELECT
        *,
        CASE
            WHEN previous_date = date
            THEN 1 ELSE 0
        END AS same_date,

        CASE
            WHEN previous_pos_text = pos_text
            THEN 1 ELSE 0
        END AS same_pos,

        CASE
            WHEN previous_date = date
             AND previous_pos_text = pos_text
            THEN 1 ELSE 0
        END AS same_date_and_pos,

        CASE
            WHEN previous_date = date
             AND previous_course = course
             AND previous_off = off
             AND previous_race_name = race_name
            THEN 1 ELSE 0
        END AS same_provisional_race
    FROM ordered_rows
    WHERE previous_date IS NOT NULL
)
SELECT
    COUNT(*) AS adjacent_row_pairs,
    SUM(same_date) AS pairs_with_same_date,
    ROUND(100.0 * SUM(same_date) / COUNT(*), 2) AS pct_same_date,

    SUM(same_pos) AS pairs_with_same_pos,
    ROUND(100.0 * SUM(same_pos) / COUNT(*), 2) AS pct_same_pos,

    SUM(same_date_and_pos) AS pairs_with_same_date_and_pos,
    ROUND(
        100.0 * SUM(same_date_and_pos) / COUNT(*),
        2
    ) AS pct_same_date_and_pos,

    SUM(same_provisional_race) AS pairs_with_same_race,
    ROUND(
        100.0 * SUM(same_provisional_race) / COUNT(*),
        2
    ) AS pct_same_race
FROM adjacent_pairs
"""

source_order_adjacency = pd.read_sql_query(
    source_order_adjacency_sql,
    connection,
)

source_order_adjacency

,adjacent_row_pairs,pairs_with_same_date,pct_same_date,pairs_with_same_pos,pct_same_pos,pairs_with_same_date_and_pos,pct_same_date_and_pos,pairs_with_same_race,pct_same_race
0,1851284,1847155,99.78,59518,3.21,59258,3.2,1354754,73.18


In [21]:
# Profile the physical ordering of source dates.
#
# We test:
# - whether each date occupies one contiguous rowid block;
# - how often the date changes between adjacent rows;
# - whether those changes move forwards or backwards in calendar order.
#
# This establishes what source rowid can preserve as lineage evidence without
# assuming that races themselves are stored contiguously.

date_order_profile_sql = f"""
WITH ordered_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        LAG(date) OVER (ORDER BY rowid) AS previous_date
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
date_transitions AS (
    SELECT
        source_rowid,
        date,
        previous_date
    FROM ordered_rows
    WHERE previous_date IS NOT NULL
      AND date <> previous_date
),
date_segments AS (
    SELECT
        date,
        1 + SUM(
            CASE
                WHEN previous_date IS NOT NULL
                 AND previous_date <> date
                THEN 1
                ELSE 0
            END
        ) AS segment_marker
    FROM ordered_rows
),
date_segment_counts AS (
    SELECT
        date,
        COUNT(*) AS physical_segments
    FROM (
        SELECT
            date,
            source_rowid,
            CASE
                WHEN previous_date IS NULL
                  OR previous_date <> date
                THEN 1
                ELSE 0
            END AS starts_new_segment
        FROM ordered_rows
    )
    GROUP BY date
)
SELECT
    (SELECT COUNT(DISTINCT date) FROM ordered_rows) AS distinct_dates,

    (SELECT COUNT(*) FROM date_transitions) AS date_changes,

    (SELECT COUNT(*)
     FROM date_transitions
     WHERE date > previous_date) AS forward_date_changes,

    (SELECT COUNT(*)
     FROM date_transitions
     WHERE date < previous_date) AS backward_date_changes,

    (SELECT COUNT(*)
     FROM date_segment_counts
     WHERE physical_segments = 1) AS dates_in_one_physical_block,

    (SELECT COUNT(*)
     FROM date_segment_counts
     WHERE physical_segments > 1) AS dates_in_multiple_physical_blocks,

    (SELECT MAX(physical_segments)
     FROM date_segment_counts) AS maximum_blocks_for_one_date
"""

date_order_profile = pd.read_sql_query(
    date_order_profile_sql,
    connection,
)

date_order_profile

,distinct_dates,date_changes,forward_date_changes,backward_date_changes,dates_in_one_physical_block,dates_in_multiple_physical_blocks,maximum_blocks_for_one_date
0,4130,4129,4129,0,0,4130,1094


In [22]:
# Correct the date-segment calculation from the preceding cell.
#
# A new date segment begins at the first data row or whenever the date differs
# from the immediately preceding physical row. We must SUM those segment-start
# markers within each date, rather than count all rows belonging to the date.
#
# Given the earlier transition results, we expect every date to occupy exactly
# one contiguous physical block in chronological order.

corrected_date_order_profile_sql = f"""
WITH ordered_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        LAG(date) OVER (ORDER BY rowid) AS previous_date
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
marked_rows AS (
    SELECT
        source_rowid,
        date,
        previous_date,
        CASE
            WHEN previous_date IS NULL
              OR previous_date <> date
            THEN 1
            ELSE 0
        END AS starts_new_date_segment
    FROM ordered_rows
),
date_segment_counts AS (
    SELECT
        date,
        SUM(starts_new_date_segment) AS physical_segments
    FROM marked_rows
    GROUP BY date
),
date_transitions AS (
    SELECT
        date,
        previous_date
    FROM marked_rows
    WHERE previous_date IS NOT NULL
      AND date <> previous_date
)
SELECT
    (SELECT COUNT(*) FROM date_segment_counts) AS distinct_dates,

    (SELECT COUNT(*) FROM date_transitions) AS date_changes,

    (SELECT COUNT(*)
     FROM date_transitions
     WHERE date > previous_date) AS forward_date_changes,

    (SELECT COUNT(*)
     FROM date_transitions
     WHERE date < previous_date) AS backward_date_changes,

    (SELECT COUNT(*)
     FROM date_segment_counts
     WHERE physical_segments = 1) AS dates_in_one_physical_block,

    (SELECT COUNT(*)
     FROM date_segment_counts
     WHERE physical_segments > 1) AS dates_in_multiple_physical_blocks,

    (SELECT MAX(physical_segments)
     FROM date_segment_counts) AS maximum_blocks_for_one_date
"""

corrected_date_order_profile = pd.read_sql_query(
    corrected_date_order_profile_sql,
    connection,
)

corrected_date_order_profile

,distinct_dates,date_changes,forward_date_changes,backward_date_changes,dates_in_one_physical_block,dates_in_multiple_physical_blocks,maximum_blocks_for_one_date
0,4130,4129,4129,0,4130,0,1


In [23]:
# Measure completeness of the proposed race- and runner-identity components.
#
# Candidate race components:
#     date + course + off + race_name
#
# Candidate runner component within a race:
#     horse
#
# We count SQL NULLs, blank/whitespace-only values and populated values
# separately. Earlier profiling showed that this source generally represents
# missingness with blank strings rather than meaningful SQL NULLs.
#
# This is a completeness test only; it does not yet accept the components as
# durable keys.

identity_fields = ["date", "course", "off", "race_name", "horse"]

identity_completeness_queries = []

for field in identity_fields:
    quoted_field = f'"{field}"'

    identity_completeness_queries.append(
        f"""
        SELECT
            '{field}' AS field,
            COUNT(*) AS data_rows,
            SUM(CASE WHEN {quoted_field} IS NULL THEN 1 ELSE 0 END) AS sql_null_rows,
            SUM(
                CASE
                    WHEN {quoted_field} IS NOT NULL
                     AND TRIM(CAST({quoted_field} AS TEXT)) = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_rows,
            SUM(
                CASE
                    WHEN {quoted_field} IS NOT NULL
                     AND TRIM(CAST({quoted_field} AS TEXT)) <> ''
                    THEN 1
                    ELSE 0
                END
            ) AS populated_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """
    )

identity_completeness_sql = "\nUNION ALL\n".join(identity_completeness_queries)

identity_completeness = pd.read_sql_query(
    identity_completeness_sql,
    connection,
)

identity_completeness

,field,data_rows,sql_null_rows,blank_rows,populated_rows
0,date,1851285,0,0,1851285
1,course,1851285,0,0,1851285
2,off,1851285,0,0,1851285
3,race_name,1851285,0,0,1851285
4,horse,1851285,0,0,1851285


In [24]:
# Test whether simple text normalisation changes race or runner identity counts.
#
# We compare the raw candidate groupings with versions that:
# - trim leading and trailing whitespace;
# - convert text to lower case.
#
# If normalisation reduces the number of distinct groups, some currently
# separate identities differ only by case or outer whitespace. That would
# indicate a need for canonical comparison fields while still preserving the
# original source text.
#
# Internal punctuation, spacing and spelling are deliberately left unchanged.

normalised_identity_summary_sql = f"""
WITH identity_counts AS (
    SELECT
        COUNT(*) AS data_rows,

        COUNT(*) OVER () AS unused_window_value
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    LIMIT 1
)
SELECT
    (SELECT COUNT(*)
     FROM (
         SELECT
             date,
             course,
             off,
             race_name
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY
             date,
             course,
             off,
             race_name
     )) AS raw_race_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT
             TRIM(CAST(date AS TEXT)) AS date_key,
             LOWER(TRIM(CAST(course AS TEXT))) AS course_key,
             TRIM(CAST(off AS TEXT)) AS off_key,
             LOWER(TRIM(CAST(race_name AS TEXT))) AS race_name_key
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY
             date_key,
             course_key,
             off_key,
             race_name_key
     )) AS normalised_race_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT
             date,
             course,
             off,
             race_name,
             horse
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY
             date,
             course,
             off,
             race_name,
             horse
     )) AS raw_runner_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT
             TRIM(CAST(date AS TEXT)) AS date_key,
             LOWER(TRIM(CAST(course AS TEXT))) AS course_key,
             TRIM(CAST(off AS TEXT)) AS off_key,
             LOWER(TRIM(CAST(race_name AS TEXT))) AS race_name_key,
             LOWER(TRIM(CAST(horse AS TEXT))) AS horse_key
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY
             date_key,
             course_key,
             off_key,
             race_name_key,
             horse_key
     )) AS normalised_runner_groups
"""

normalised_identity_summary = pd.read_sql_query(
    normalised_identity_summary_sql,
    connection,
)

normalised_identity_summary["race_groups_merged"] = (
    normalised_identity_summary["raw_race_groups"]
    - normalised_identity_summary["normalised_race_groups"]
)

normalised_identity_summary["runner_groups_merged"] = (
    normalised_identity_summary["raw_runner_groups"]
    - normalised_identity_summary["normalised_runner_groups"]
)

normalised_identity_summary

,raw_race_groups,normalised_race_groups,raw_runner_groups,normalised_runner_groups,race_groups_merged,runner_groups_merged
0,189043,189043,1851285,1851285,0,0


In [25]:
# Compare reduced descriptive race-key candidates.
#
# The full provisional grouping is:
#     date + course + off + race_name
#
# We count distinct groups produced by several reduced combinations. A reduced
# candidate matching 189,043 groups has no collisions in the current extract;
# a lower count means that combination merges separate provisional races.
#
# This tests empirical distinctness only. It does not mean a shorter grouping
# is semantically preferable or safe for future source extracts.

race_key_component_summary_sql = f"""
SELECT
    (SELECT COUNT(*)
     FROM (
         SELECT date, course, off, race_name
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, course, off, race_name
     )) AS date_course_off_name_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, course, off
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, course, off
     )) AS date_course_off_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, course, race_name
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, course, race_name
     )) AS date_course_name_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, off, race_name
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, off, race_name
     )) AS date_off_name_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, course
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, course
     )) AS date_course_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, off
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, off
     )) AS date_off_groups,

    (SELECT COUNT(*)
     FROM (
         SELECT date, race_name
         FROM {SOURCE_TABLE}
         WHERE {DATA_ROW_PREDICATE}
         GROUP BY date, race_name
     )) AS date_name_groups
"""

race_key_component_summary = pd.read_sql_query(
    race_key_component_summary_sql,
    connection,
)

race_key_component_summary

,date_course_off_name_groups,date_course_off_groups,date_course_name_groups,date_off_name_groups,date_course_groups,date_off_groups,date_name_groups
0,189043,189043,188527,189043,33146,177829,187672


In [26]:
# Inspect collisions created by the reduced key:
#     date + course + race_name
#
# These groups contain more than one off-time, so omitting `off` would merge
# distinct provisional races. We show the largest collision groups first and
# retain each supplied race_id and runner count for interpretation.

date_course_name_collisions_sql = f"""
WITH reduced_key_profiles AS (
    SELECT
        date,
        course,
        race_name,
        COUNT(DISTINCT off) AS distinct_off_times,
        COUNT(*) AS total_runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        race_name
    HAVING COUNT(DISTINCT off) > 1
),
race_groups AS (
    SELECT
        date,
        course,
        race_name,
        off,
        CAST(race_id AS TEXT) AS race_id,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        race_name,
        off,
        CAST(race_id AS TEXT)
)
SELECT
    p.date,
    p.course,
    p.race_name,
    p.distinct_off_times,
    p.total_runner_rows,
    r.off,
    r.race_id,
    r.runner_rows
FROM reduced_key_profiles AS p
JOIN race_groups AS r
    ON r.date = p.date
   AND r.course = p.course
   AND r.race_name = p.race_name
ORDER BY
    p.distinct_off_times DESC,
    p.date,
    p.course,
    p.race_name,
    r.off
LIMIT 100
"""

date_course_name_collisions = pd.read_sql_query(
    date_course_name_collisions_sql,
    connection,
)

print(
    "Date/course/race-name groups containing multiple off-times:",
    date_course_name_collisions[
        ["date", "course", "race_name"]
    ].drop_duplicates().shape[0],
    "(first 100 race rows displayed)",
)

date_course_name_collisions

Date/course/race-name groups containing multiple off-times: 27 (first 100 race rows displayed)


,date,course,race_name,distinct_off_times,total_runner_rows,off,race_id,runner_rows
0,2021-03-15,Punchestown (IRE),Punchestown (C & G) Point-To-Point INH Flat Race,6,62,2:25,780053,9
1,2021-03-15,Punchestown (IRE),Punchestown (C & G) Point-To-Point INH Flat Race,6,62,2:55,780054,11
2,2021-03-15,Punchestown (IRE),Punchestown (C & G) Point-To-Point INH Flat Race,6,62,3:25,780055,6
3,2021-03-15,Punchestown (IRE),Punchestown (C & G) Point-To-Point INH Flat Race,6,62,3:55,780056,12
4,2021-03-15,Punchestown (IRE),Punchestown (C & G) Point-To-Point INH Flat Race,6,62,4:30,780057,9
...,...,...,...,...,...,...,...,...
95,2020-04-27,Will Rogers Downs (USA),Claiming Race (3yo+) (Dirt),3,31,9:45,756594,10
96,2020-04-28,Will Rogers Downs (USA),Maiden Claiming Race (Maiden) (3yo+) (Dirt),3,28,7:15,756626,8
97,2020-04-28,Will Rogers Downs (USA),Maiden Claiming Race (Maiden) (3yo+) (Dirt),3,28,9:15,756631,9
98,2020-04-28,Will Rogers Downs (USA),Maiden Claiming Race (Maiden) (3yo+) (Dirt),3,28,9:45,756632,11


In [27]:
# Quantify all collisions created by omitting `off` from the race key.
#
# For each date + course + race_name group, count how many distinct off-times
# it contains. We then report:
# - the number of reduced-key groups that collide;
# - the number of actual races contained in those groups;
# - the number of races lost through merging;
# - the maximum number of same-named races at one meeting.
#
# "Races lost through merging" is calculated as:
#     sum(distinct_off_times - 1)

date_course_name_collision_summary_sql = f"""
WITH reduced_key_profiles AS (
    SELECT
        date,
        course,
        race_name,
        COUNT(DISTINCT off) AS distinct_off_times,
        COUNT(*) AS runner_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        date,
        course,
        race_name
    HAVING COUNT(DISTINCT off) > 1
)
SELECT
    COUNT(*) AS colliding_reduced_key_groups,
    SUM(distinct_off_times) AS races_inside_collision_groups,
    SUM(distinct_off_times - 1) AS races_lost_if_off_omitted,
    MAX(distinct_off_times) AS maximum_same_named_races_at_one_meeting,
    SUM(runner_rows) AS runner_rows_affected
FROM reduced_key_profiles
"""

date_course_name_collision_summary = pd.read_sql_query(
    date_course_name_collision_summary_sql,
    connection,
)

date_course_name_collision_summary

,colliding_reduced_key_groups,races_inside_collision_groups,races_lost_if_off_omitted,maximum_same_named_races_at_one_meeting,runner_rows_affected
0,451,967,516,6,10410


In [28]:
# Profile SQLite rowid as a source-lineage field.
#
# The raw table contains one imported header row at rowid = 1, which is
# excluded by DATA_ROW_PREDICATE. For the remaining data-like rows, we test:
# - whether every rowid is unique;
# - the minimum and maximum rowid;
# - whether the data-row sequence contains physical gaps;
# - whether row count matches the expected rowid span after excluding rowid 1.
#
# rowid is not proposed as a business key. Its role would be to preserve an
# exact pointer back to the original physical source record.

source_rowid_profile_sql = f"""
SELECT
    COUNT(*) AS data_rows,
    COUNT(DISTINCT rowid) AS distinct_source_rowids,
    MIN(rowid) AS minimum_source_rowid,
    MAX(rowid) AS maximum_source_rowid,
    MAX(rowid) - MIN(rowid) + 1 AS source_rowid_span,
    (MAX(rowid) - MIN(rowid) + 1) - COUNT(*) AS missing_rowids_inside_data_span
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
"""

source_rowid_profile = pd.read_sql_query(
    source_rowid_profile_sql,
    connection,
)

source_rowid_profile

,data_rows,distinct_source_rowids,minimum_source_rowid,maximum_source_rowid,source_rowid_span,missing_rowids_inside_data_span
0,1851285,1851285,2,1851286,1851285,0


In [29]:
# Consolidate the principal identity findings into one validation table.
#
# This cell does not create identifiers or design the target schema. It records
# the observed evidence supporting or rejecting each source-key candidate.
#
# The resulting table will make the distinction clear between:
# - uniqueness in the current extract;
# - semantic suitability as an identity rule;
# - and source-lineage usefulness.

identity_evidence = pd.DataFrame(
    [
        {
            "candidate_or_field": "race_id",
            "observed_groups_or_rows": 188_782,
            "collision_or_reuse_evidence": "206 values occur on multiple dates",
            "current_extract_unique": False,
            "provisional_role": "Preserve as a source attribute only",
        },
        {
            "candidate_or_field": "date + race_id",
            "observed_groups_or_rows": 189_035,
            "collision_or_reuse_evidence": "8 pairs each describe two different races",
            "current_extract_unique": False,
            "provisional_role": "Insufficient race identity",
        },
        {
            "candidate_or_field": "date + course + off",
            "observed_groups_or_rows": 189_043,
            "collision_or_reuse_evidence": "No collisions observed",
            "current_extract_unique": True,
            "provisional_role": "Leading candidate race identity",
        },
        {
            "candidate_or_field": "date + course + off + race_name",
            "observed_groups_or_rows": 189_043,
            "collision_or_reuse_evidence": "No collisions observed",
            "current_extract_unique": True,
            "provisional_role": "Conservative provisional race grouping",
        },
        {
            "candidate_or_field": "race grouping + horse",
            "observed_groups_or_rows": 1_851_285,
            "collision_or_reuse_evidence": "No duplicate horse within a provisional race",
            "current_extract_unique": True,
            "provisional_role": "Leading candidate runner identity",
        },
        {
            "candidate_or_field": "race grouping + num",
            "observed_groups_or_rows": None,
            "collision_or_reuse_evidence": "700 duplicate-number groups",
            "current_extract_unique": False,
            "provisional_role": "Preserve as racecard/betting-entry attribute",
        },
        {
            "candidate_or_field": "full supplied row contents",
            "observed_groups_or_rows": 1_851_285,
            "collision_or_reuse_evidence": "No exact duplicate source records",
            "current_extract_unique": True,
            "provisional_role": "Evidence against copied duplicate rows",
        },
        {
            "candidate_or_field": "source rowid",
            "observed_groups_or_rows": 1_851_285,
            "collision_or_reuse_evidence": "Unique contiguous range 2–1,851,286",
            "current_extract_unique": True,
            "provisional_role": "Source-lineage pointer, not business identity",
        },
    ]
)

identity_evidence

,candidate_or_field,observed_groups_or_rows,collision_or_reuse_evidence,current_extract_unique,provisional_role
0,race_id,188782.0,206 values occur on multiple dates,False,Preserve as a source attribute only
1,date + race_id,189035.0,8 pairs each describe two different races,False,Insufficient race identity
2,date + course + off,189043.0,No collisions observed,True,Leading candidate race identity
3,date + course + off + race_name,189043.0,No collisions observed,True,Conservative provisional race grouping
4,race grouping + horse,1851285.0,No duplicate horse within a provisional race,True,Leading candidate runner identity
5,race grouping + num,NaN,700 duplicate-number groups,False,Preserve as racecard/betting-entry attribute
6,full supplied row contents,1851285.0,No exact duplicate source records,True,Evidence against copied duplicate rows
7,source rowid,1851285.0,"Unique contiguous range 2–1,851,286",True,"Source-lineage pointer, not business identity"


## Interim identity findings

### Observations

1. The supplied `race_id` is not globally unique.

   - 188,782 distinct values occur across 189,043 provisional races.
   - 206 `race_id` values occur on more than one date.
   - One identifier occurs on five separate dates.
   - The repeated identifiers refer to unrelated races across different dates, courses and jurisdictions.

2. Adding `date` does not make `race_id` reliable.

   - There are 189,035 distinct `date + race_id` combinations.
   - Eight combinations each contain two clearly different races.
   - These collisions include different courses or different races at the same course.

3. The descriptive race grouping is internally consistent.

   - `date + course + off + race_name` produces 189,043 groups.
   - No descriptive group contains multiple `race_id` values.
   - No `date + course + off` combination contains multiple race names.
   - `date + course + off` therefore also produces 189,043 groups in this extract.
   - Every identity component is populated on every data-like row.
   - Trimming outer whitespace and ignoring case does not merge any race groups.

4. Off-time is necessary for race identity.

   - Omitting `off` creates 451 colliding `date + course + race_name` groups.
   - Those groups contain 967 actual races.
   - Merging them would lose 516 races and affect 10,410 runner rows.
   - Up to six races at one meeting share the same supplied race name.

5. Horse name is unique within the provisional race grouping.

   - `date + course + off + race_name + horse` produces 1,851,285 groups.
   - No horse occurs twice within one provisional race.
   - No horse appears in multiple provisional races at the same course on the same date.

6. Runner number is not an individual-runner identifier.

   - 700 provisional race-and-number groups contain multiple horses.
   - 177 groups use `num = 0`, covering 1,170 rows.
   - 523 groups share a non-zero number, covering 1,084 rows.
   - Many are consistent with coupled betting entries, but some shared numbers have different prices, draws and trainers.
   - `num` therefore has jurisdiction-dependent meaning and cannot be interpreted uniformly.

7. No straightforward repeated source records were found.

   - No two rows are identical across every supplied source column.
   - No provisional race-and-horse group contains multiple source rows.
   - There is therefore no evidence of exact copied rows or repeated runner versions under the same current identity fields.

8. SQLite `rowid` preserves useful source lineage.

   - Data-like records occupy the unique uninterrupted range `2` to `1,851,286`.
   - `rowid = 1` is the imported header row.
   - All dates occupy one chronological physical block.
   - Races within a date may be split across as many as 11 physical segments.
   - Physical adjacency cannot therefore be used to reconstruct race membership.

### Provisional interpretation

The leading candidate race identity for the current source is:

`date + course + off`

For conservative source reconstruction, the working race grouping should remain:

`date + course + off + race_name`

The additional `race_name` component does not change the group count in the current extract, but it provides a useful descriptive validation field and protection against future extracts containing same-slot anomalies.

The leading candidate runner identity within a reconstructed race is:

`race identity + horse`

These are natural-key candidates for matching and validation, not suitable permanent database identifiers. A later staging layer should assign independent surrogate race and runner identifiers while retaining all natural-key components and source-lineage fields.

The supplied `race_id`, `num` and SQLite `rowid` should all be preserved, but for different purposes:

- `race_id`: supplied source reference with known reuse;
- `num`: jurisdiction-dependent racecard or betting-entry attribute;
- `rowid`: exact physical source-row lineage within this immutable database extract.

### Remaining risks

- A future source extract may revise a race name, advertised off-time or course spelling.
- Two genuinely different races could theoretically share the same date, course and advertised off-time in data from another provider or jurisdiction.
- Horse names are sufficient for runner-record identity in this extract but do not constitute durable horse-entity identity across races.
- Coupled-entry structure cannot be reconstructed reliably from `num` alone.
- Near-duplicate or amended records with changed identity fields cannot be ruled out solely by the absence of exact duplicates.

In [30]:
# Record the provisional identity and lineage recommendations in a structured
# table for later use in the notebook report and closeout metadata.
#
# These are recommendations derived from the observed evidence. They do not
# create the staging schema or assign any identifiers.

identity_recommendations = pd.DataFrame(
    [
        {
            "area": "Race matching",
            "recommendation": "Use date + course + off as the leading natural race identity",
            "status": "Candidate",
            "reason": "Produces 189,043 unique groups with no observed collisions",
        },
        {
            "area": "Race validation",
            "recommendation": "Retain race_name alongside the candidate race identity",
            "status": "Required source attribute",
            "reason": "Provides descriptive validation and protection against future same-slot anomalies",
        },
        {
            "area": "Runner matching",
            "recommendation": "Use candidate race identity + horse as the leading natural runner identity",
            "status": "Candidate",
            "reason": "Produces one unique group for every data-like source row",
        },
        {
            "area": "Race surrogate",
            "recommendation": "Assign an independent staging race identifier",
            "status": "Required later",
            "reason": "Natural descriptive values may be corrected or reformatted in later extracts",
        },
        {
            "area": "Runner surrogate",
            "recommendation": "Assign an independent staging runner identifier linked to the staging race identifier",
            "status": "Required later",
            "reason": "Horse text identifies a source record but is not durable horse-entity identity",
        },
        {
            "area": "Source race reference",
            "recommendation": "Preserve race_id without enforcing uniqueness",
            "status": "Required lineage",
            "reason": "The supplied value is reused across dates and within eight date-specific collisions",
        },
        {
            "area": "Runner number",
            "recommendation": "Preserve num without using it as a runner key",
            "status": "Required source attribute",
            "reason": "Zero sentinels and shared jurisdiction-dependent betting numbers create 700 collisions",
        },
        {
            "area": "Physical source lineage",
            "recommendation": "Preserve source database identity, source table and original rowid",
            "status": "Required lineage",
            "reason": "Together they provide an exact pointer back to the immutable source record",
        },
        {
            "area": "Source text",
            "recommendation": "Preserve raw date, course, off, race_name and horse values",
            "status": "Required lineage",
            "reason": "Canonical matching fields must not replace the original supplied text",
        },
        {
            "area": "Future ingestion",
            "recommendation": "Re-run collision and uniqueness checks for every new source snapshot",
            "status": "Required control",
            "reason": "Current-extract uniqueness does not guarantee stability across amended or extended data",
        },
    ]
)

identity_recommendations

,area,recommendation,status,reason
0,Race matching,Use date + course + off as the leading natural...,Candidate,"Produces 189,043 unique groups with no observe..."
1,Race validation,Retain race_name alongside the candidate race ...,Required source attribute,Provides descriptive validation and protection...
2,Runner matching,Use candidate race identity + horse as the lea...,Candidate,Produces one unique group for every data-like ...
3,Race surrogate,Assign an independent staging race identifier,Required later,Natural descriptive values may be corrected or...
4,Runner surrogate,Assign an independent staging runner identifie...,Required later,Horse text identifies a source record but is n...
5,Source race reference,Preserve race_id without enforcing uniqueness,Required lineage,The supplied value is reused across dates and ...
6,Runner number,Preserve num without using it as a runner key,Required source attribute,Zero sentinels and shared jurisdiction-depende...
7,Physical source lineage,"Preserve source database identity, source tabl...",Required lineage,Together they provide an exact pointer back to...
8,Source text,"Preserve raw date, course, off, race_name and ...",Required lineage,Canonical matching fields must not replace the...
9,Future ingestion,Re-run collision and uniqueness checks for eve...,Required control,Current-extract uniqueness does not guarantee ...


## Notebook conclusion

### Answer to the bounded question

A race can be identified reliably within the current source by using the descriptive meeting slot:

`date + course + off`

This combination produces 189,043 distinct race groups and has no observed collisions in the current extract.

For conservative source reconstruction, `race_name` should remain attached to the grouping:

`date + course + off + race_name`

Although `race_name` does not create any additional groups in this extract, it provides a descriptive validation field and may expose anomalies in future or amended snapshots.

A runner record can be identified within a reconstructed race by adding the supplied horse name:

`date + course + off + race_name + horse`

This combination is complete and unique across all 1,851,285 data-like source rows.

These combinations are suitable as **candidate natural matching rules**, not as permanent database identifiers.

### Why the supplied identifiers are insufficient

The supplied `race_id` cannot serve as a globally unique race key:

- 206 values occur on multiple dates;
- some values are reused for unrelated races across courses and jurisdictions;
- eight `date + race_id` combinations each refer to two different races.

The supplied `num` cannot serve as an individual-runner key:

- 700 race-and-number groups contain multiple horses;
- `0` frequently acts as an unavailable-number sentinel;
- non-zero values may represent coupled betting entries;
- some duplicate non-zero values do not behave consistently enough to support one universal interpretation.

### Repeated and amended records

No exact repeated records were found:

- no duplicate rows exist across all supplied columns;
- no horse occurs twice within one provisional race;
- no horse occurs in multiple provisional races at the same course on the same date.

This provides no evidence of straightforward copied or amended runner rows under the current identity fields. It does not rule out revisions that altered the descriptive race fields or horse text.

### Source-lineage requirement

The later ingestion process should preserve enough information to trace every staging record back to the immutable source:

- source product or database identity;
- source file or database path;
- source table name;
- original SQLite `rowid`;
- supplied `race_id`;
- supplied `num`;
- original unmodified identity text, including `date`, `course`, `off`, `race_name` and `horse`.

SQLite `rowid` is unique and uninterrupted from 2 through 1,851,286 for the data-like rows. It is therefore a precise source-row pointer within this extract, but it is not a durable business key and may change if the database is rebuilt.

### Surrogate identifiers required later

A later staging layer will need:

1. an independent surrogate race identifier assigned to each reconstructed race;
2. an independent surrogate runner-record identifier assigned to each source runner row;
3. a relationship from each runner record to its reconstructed race;
4. retained candidate natural-key fields for validation and repeatable matching;
5. retained source-lineage fields for audit, debugging and re-ingestion.

No permanent horse-entity identifier is justified yet. The supplied horse name identifies a runner record within a race, but horse entity resolution across races is a separate problem.

### Unresolved identity risks

- Advertised off-times can potentially be corrected in later snapshots.
- Course and race-name text may be reformatted or renamed.
- Future data may contain two races sharing the same descriptive meeting slot.
- Horse-name spelling may change across sources or snapshots.
- Coupled-entry structure cannot be reconstructed reliably from `num` alone.
- Current-extract uniqueness must therefore be revalidated whenever the source is extended, replaced or refreshed.

### Final provisional rules

**Candidate race matching rule**

`date + course + off`

with `race_name` retained as a required descriptive validation field.

**Candidate runner-record matching rule**

`candidate race identity + horse`

with `num` retained only as a source racecard or betting-entry attribute.

**Permanent identifiers**

Use staging-generated surrogate race and runner-record identifiers rather than treating any supplied field or descriptive combination as an immutable primary key.

In [31]:
# Validate the principal Notebook 03 findings before closeout.
#
# These assertions convert the notebook's key observations into reproducible
# controls. A failed assertion means either the source changed or an earlier
# conclusion needs to be revisited.
#
# This cell does not modify the source database.

validation_checks = {
    "data_rows": int(source_rowid_profile.loc[0, "data_rows"]),
    "provisional_races": int(
        normalised_identity_summary.loc[0, "raw_race_groups"]
    ),
    "distinct_race_ids": int(
        race_id_summary.loc[0, "distinct_race_ids"]
    ),
    "race_ids_used_on_multiple_dates": int(
        race_id_summary.loc[0, "race_ids_used_on_multiple_dates"]
    ),
    "race_id_date_pairs": int(
        race_id_summary.loc[0, "distinct_race_id_date_pairs"]
    ),
    "race_id_date_collision_pairs": int(
        race_id_date_collisions[
            ["race_id", "date"]
        ].drop_duplicates().shape[0]
    ),
    "duplicate_horse_within_race_groups": int(
        len(duplicate_horse_within_race)
    ),
    "duplicate_runner_number_groups": int(
        len(duplicate_num_within_race)
    ),
    "exact_duplicate_groups": int(
        exact_duplicate_summary.loc[0, "exact_duplicate_groups"]
    ),
    "distinct_source_rowids": int(
        source_rowid_profile.loc[0, "distinct_source_rowids"]
    ),
    "dates_in_multiple_physical_blocks": int(
        corrected_date_order_profile.loc[
            0, "dates_in_multiple_physical_blocks"
        ]
    ),
}

assert validation_checks["data_rows"] == 1_851_285
assert validation_checks["provisional_races"] == 189_043
assert validation_checks["distinct_race_ids"] == 188_782
assert validation_checks["race_ids_used_on_multiple_dates"] == 206
assert validation_checks["race_id_date_pairs"] == 189_035
assert validation_checks["race_id_date_collision_pairs"] == 8
assert validation_checks["duplicate_horse_within_race_groups"] == 0
assert validation_checks["duplicate_runner_number_groups"] == 700
assert validation_checks["exact_duplicate_groups"] == 0
assert validation_checks["distinct_source_rowids"] == 1_851_285
assert validation_checks["dates_in_multiple_physical_blocks"] == 0

validation_results = pd.DataFrame(
    {
        "check": validation_checks.keys(),
        "observed_value": validation_checks.values(),
        "status": "PASS",
    }
)

print("Notebook 03 identity validations passed.")
validation_results

Notebook 03 identity validations passed.


,check,observed_value,status
0,data_rows,1851285,PASS
1,provisional_races,189043,PASS
2,distinct_race_ids,188782,PASS
3,race_ids_used_on_multiple_dates,206,PASS
4,race_id_date_pairs,189035,PASS
5,race_id_date_collision_pairs,8,PASS
6,duplicate_horse_within_race_groups,0,PASS
7,duplicate_runner_number_groups,700,PASS
8,exact_duplicate_groups,0,PASS
9,distinct_source_rowids,1851285,PASS


In [32]:
# Close the read-only SQLite connection now that all Notebook 03 analysis and
# validation queries have completed.
#
# Explicitly closing the connection makes the notebook lifecycle clear and
# ensures that no database handle remains open after execution.

connection.close()

print("Read-only source connection closed.")
print("Notebook 03 analysis and validation complete.")

Read-only source connection closed.
Notebook 03 analysis and validation complete.


## Validation status

Notebook 03 completed successfully from the current kernel.

All principal controls passed:

- 1,851,285 data-like source rows were analysed;
- 189,043 provisional races were reconstructed;
- 206 supplied `race_id` values were confirmed as reused across dates;
- eight `date + race_id` combinations were confirmed to contain two different races;
- no duplicate horse records were found within a provisional race;
- 700 duplicated runner-number groups were confirmed;
- no exact duplicate source records were found;
- every data row retained a unique source `rowid`;
- every source date occupied one chronological physical block.

The read-only SQLite connection was closed explicitly after validation.

**Notebook status:** analysis complete and ready for clean-kernel execution and repository closeout.